# 03 Train BayesFlow

This notebook trains the BayesFlow model using the gravitational-wave dataset created in Notebook 02.

The dataset contains:
- `X`: the noisy and whitened signals
- `theta`: the six true parameter values
- Parameter order: `[m1, m2, chi1, chi2, distance, inclination]`

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
drive_root = Path("/content/drive/MyDrive/GW_Project")
local_root = cwd.parent if cwd.name == "notebooks" else cwd
PROJECT_ROOT = drive_root if (drive_root / "src").exists() else local_root
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError(
        f"Could not find the project from {cwd}. In Colab, mount Drive first; "
        f"expected the repository at {drive_root}."
    )
project_path = str(PROJECT_ROOT)
if project_path in sys.path:
    sys.path.remove(project_path)
sys.path.insert(0, project_path)

DATASET_SIZE = 25000
SIGNAL_LENGTH = 2048
DATA_DIR = PROJECT_ROOT / "data"
DATASET_PATH = DATA_DIR / f"gw_dataset_{DATASET_SIZE}.npz"
MODEL_DIR = PROJECT_ROOT / "models" / f"bayesflow_model_{DATASET_SIZE}"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.model import (
    PARAMETER_NAMES,
    build_workflow,
    history_to_losses,
    load_npz_dataset,
    sample_posterior,
    split_dataset_three_way,
)

# Notebook 02 saves the signals and parameters separately.
# Combine them into one file before training.
if not DATASET_PATH.exists():
    waveforms_path = DATA_DIR / f"waveforms_{DATASET_SIZE}.npy"
    parameters_path = DATA_DIR / f"parameters_{DATASET_SIZE}.npy"

    if not waveforms_path.exists() or not parameters_path.exists():
        raise FileNotFoundError(
            f"Could not find {DATASET_PATH.name}, {waveforms_path.name}, or {parameters_path.name}. "
            "Run notebooks/02_generate_dataset.ipynb first."
        )

    X = np.load(waveforms_path, mmap_mode="r")
    theta = np.load(parameters_path, mmap_mode="r")
    expected_x_shape = (DATASET_SIZE, SIGNAL_LENGTH)
    expected_theta_shape = (DATASET_SIZE, 6)

    if X.shape != expected_x_shape or theta.shape != expected_theta_shape:
        raise ValueError(
            f"Unexpected notebook 02 output shapes: X={X.shape}, theta={theta.shape}; "
            f"expected {expected_x_shape} and {expected_theta_shape}."
        )
    if not np.isfinite(X).all() or not np.isfinite(theta).all():
        raise ValueError("Generated dataset contains NaN or infinite values.")
    if not np.all(theta[:, 0] >= theta[:, 1]):
        raise ValueError("Generated dataset violates the required ordering m1 >= m2.")

    np.savez(DATASET_PATH, X=X, theta=theta)
    print("Created training dataset:", DATASET_PATH)
    del X, theta

dataset = load_npz_dataset(DATASET_PATH)
print("strain:", dataset["strain"].shape)
print("parameters:", dataset["parameters"].shape)
print("parameter order:", PARAMETER_NAMES)

## Training

The 25,000 simulations are split into 20,000 training, 2,500 validation, and 2,500 test examples. The test set is not used during training.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64
VALIDATION_FRACTION = 0.10
TEST_FRACTION = 0.10
SEED = 2026

train_data, val_data, test_data = split_dataset_three_way(
    dataset,
    validation_fraction=VALIDATION_FRACTION,
    test_fraction=TEST_FRACTION,
    seed=SEED,
)

workflow = build_workflow(model_dir=MODEL_DIR)
history = workflow.fit_offline(
    data=train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
)

print("train strain:", train_data["strain"].shape)
print("validation strain:", val_data["strain"].shape)
print("test strain:", test_data["strain"].shape)

In [ ]:
train_loss, val_loss = history_to_losses(history)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_loss, marker="o", label="train loss")
if val_loss is not None:
    ax.plot(val_loss, marker="o", label="validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("BayesFlow training loss")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()

loss_path = FIGURE_DIR / "training_loss.png"
fig.savefig(loss_path, dpi=160)
loss_path

## Posterior Example

Use one held-out test signal to check the model predictions. The dashed lines show the true values.

In [ ]:
test_strain = test_data["strain"][0, :, 0]
true_theta = test_data["parameters"][0]
posterior_samples = sample_posterior(workflow, test_strain, num_samples=1000)

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.ravel()

for i, name in enumerate(PARAMETER_NAMES):
    axes[i].hist(posterior_samples[:, i], bins=35, density=True, alpha=0.75)
    axes[i].axvline(true_theta[i], color="black", linestyle="--", label="true")
    axes[i].set_title(name)
    axes[i].grid(alpha=0.2)

axes[0].legend()
fig.suptitle("Posterior samples for one test signal")
fig.tight_layout()

posterior_path = FIGURE_DIR / "posterior_example.png"
fig.savefig(posterior_path, dpi=160)
posterior_path

In [ ]:
posterior_mean = posterior_samples.mean(axis=0)
posterior_std = posterior_samples.std(axis=0)
absolute_error = np.abs(posterior_mean - true_theta)

for name, truth, mean, std, err in zip(PARAMETER_NAMES, true_theta, posterior_mean, posterior_std, absolute_error):
    print(f"{name:12s} true={truth:9.4f} posterior_mean={mean:9.4f} std={std:9.4f} abs_error={err:9.4f}")

best = PARAMETER_NAMES[int(np.argmin(absolute_error))]
worst = PARAMETER_NAMES[int(np.argmax(absolute_error))]
print(f"\nFor this example, {best} is closest to the true value and {worst} has the largest error.")

## Run Summary

The final model uses the full dataset created in Notebook 02.

- Dataset: `data/gw_dataset_25000.npz`
- Dataset shape: `X = (25000, 2048)`, `theta = (25000, 6)`
- Split: 20,000 training, 2,500 validation, 2,500 test
- Epochs: `20`
- Batch size: `64`
- Saved model: `models/bayesflow_model_25000/model.keras`
- Figures: `figures/training_loss.png`, `figures/posterior_example.png`

The notebook saves the trained model, the loss graph, and one example posterior plot.
